# EDA NextWatch

Analyse exploratoire des données MovieLens 100k (notes collectées entre le 19 septembre 1997 et le 22 avril 1998).

**Question visée (produit) :** si une personne a aimé un film en 1997, quels autres films aurait-elle pu aimer ?

**Rôle de cette EDA :** préparer cette reco en comprenant le catalogue (genres, co-occurrences) et les profils (âge, sexe) qui serviront à **pondérer** les recommandations — pas encore à prédire un film.

Les analyses portent surtout sur les **parts de notes par genre** (volume d'activité), pas sur la note moyenne de satisfaction.

1. **Qu'ai-je ?** — structure (notes, users, films, genres)
2. **Puis-je faire confiance aux données ?** — qualité, manquants (dont genre `unknown`)
3. **Genres, âge, sexe** — ce qu'il faut savoir pour une reco « film aimé → autres films »
4. **Qu'est-ce qui manque ?** — biais et limites avant le modèle


## Étape 1 — Qu'ai-je ?

On pose le cadre utile à une reco : une note = un utilisateur + un film ; chaque film porte des **genres** ; chaque utilisateur a un **âge** et un **sexe**.  
On décrit dimensions, variables et types avant d'exploiter ces signaux.


In [ ]:
from pathlib import Path

import pandas as pd

# Référentiel des genres (sans "unknown" : traité comme manquant et retiré ci-dessous)
GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

_candidates = [Path("../dataset"), Path("dataset")]
DATA_DIR = next(
    (p for p in _candidates if p.is_dir() and (p / "u.data.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Dossier dataset introuvable (u.data.csv manquant).")

notes = pd.read_csv(DATA_DIR / "u.data.csv")
films = pd.read_csv(DATA_DIR / "u.item.csv")
users = pd.read_csv(DATA_DIR / "u.user.csv", dtype={"zip_code": "string"})
# occupation présent dans u.user.csv mais exclu du traitement / analyse
users = users.drop(columns=["occupation"])

notes["datetime"] = pd.to_datetime(notes["timestamp"], unit="s", utc=True)
films["release_date"] = pd.to_datetime(
    films["release_date"], format="%d-%b-%Y", errors="coerce"
)
films["release_year"] = films["release_date"].dt.year

df_raw = notes.merge(users, on="user_id", how="left").merge(
    films, left_on="item_id", right_on="movie_id", how="left"
)

# Genre "unknown" = information manquante → documenté puis sorti du jeu d'analyse
n_unknown_notes = int((df_raw["unknown"] == 1).sum())
n_unknown_films = int(films.loc[films["unknown"] == 1, "movie_id"].nunique())
UNKNOWN_MISSING = {
    "type": "genre unknown (non renseigné)",
    "n_films": n_unknown_films,
    "n_notes": n_unknown_notes,
    "pourcentage_notes": round(100 * n_unknown_notes / len(df_raw), 4),
}

df = df_raw.loc[df_raw["unknown"] != 1].drop(columns=["unknown"]).copy()
df = df.drop(columns=[c for c in ["occupation"] if c in df.columns])

# Colonnes 0/1 → une colonne liste "genres" (ex. ["Action", "Sci-Fi"])
df["genres"] = df[GENRES].apply(
    lambda row: [g for g in GENRES if row[g] == 1],
    axis=1,
)
df = df.drop(columns=GENRES)

print(
    f"df analysé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes "
    f"(retirées {n_unknown_notes} notes / {n_unknown_films} films à genre unknown)"
)
df.head()


Pour garantir la fiabilité de nos analyses, nous avons commencé par auditer la qualité des données.

Concernant les valeurs manquantes, nous avons identifié un cas particulier : le genre de film étiqueté unknown. 
Dans notre dataset, cela représentait seulement 2 films et 10 notes, soit 0,01 % du total. 

Plutôt que de conserver cette catégorie vide de sens pour notre problématique, nous l'avons documentée puis exclue du périmètre d'analyse. Cela nous permet de travailler sur un jeu de données 100 % qualifié sur les genres.


### 1. Aperçu

Le DataFrame `df` a déjà été chargé dans la cellule précédente.
Les genres ne sont plus des colonnes 0/1 : une seule colonne **`genres`** contient une **liste** de genres (ex. `['Action', 'Sci-Fi']`).
On observe ici quelques lignes et un échantillon aléatoire.


**Premières lignes :**


In [ ]:
display(df.head())


**Échantillon aléatoire :**


In [ ]:
display(df.sample(5, random_state=42))


### 2. Dimensions et variables

`df.shape` indique le nombre de lignes et de colonnes. `df.columns` permet d'identifier les variables disponibles dans la table fusionnée.

In [ ]:
print("Dimensions du DataFrame :", df.shape)


On peut en deduire quela table contient 99 990 notes d'utilisateurs, et chacune de ces notes est décrite par 33 variables.

**Variables disponibles :**


In [ ]:
display(pd.DataFrame({"variable": df.columns}))


### 3. Types et informations générales

`df.info()` et `df.dtypes` permettent de vérifier les types des colonnes et la présence éventuelle de valeurs manquantes.

**Informations générales :**


In [ ]:
df.info()


**Types des variables :**


In [ ]:
display(df.dtypes.rename("type").to_frame())


### 4. Premiers ordres de grandeur

`df.describe()` résume les variables numériques avec leur effectif, leur moyenne, leur écart-type, leurs quartiles et leurs valeurs extrêmes.

In [ ]:
# Premiers ordres de grandeur sur les variables numériques
description = df.describe().T
display(description)


### 5. Conclusion

Le DataFrame rassemble notes, utilisateurs (âge, sexe) et films avec leurs genres (`occupation` du CSV source n'est pas utilisée).  
On a les briques nécessaires pour une reco basée contenu + profil.

→ **Étape suivante :** vérifier la qualité des données avant de s'en servir.


In [ ]:
print(f"Le jeu de données contient {df.shape[0]:,} lignes et {df.shape[1]} colonnes.")


Les notes, utilisateurs et films sont fusionnés dans un même DataFrame d'analyse.


## Étape 2 — Puis-je faire confiance aux données ?

On vérifie manquants (y compris le genre `unknown` déjà filtré), doublons, catégories,
min/max/quantiles, puis la cohérence des dates, unités et types.

→ Objectif : savoir ce qui est fiable avant de répondre à la question principale (étape 3).


### 1. Point de départ

Le DataFrame `df` est déjà chargé (étape 1). On enchaîne directement sur les contrôles de qualité.


In [ ]:
print("df déjà en mémoire — dimensions :", df.shape)
df.head()


### 2. Valeurs manquantes

On combine :
- les manquants classiques (`isna`) ;
- le genre **`unknown`**, traité comme information manquante et **déjà retiré** du `df` d'analyse au chargement.


In [ ]:
# Valeurs manquantes (isna) sur le df d'analyse
missing = (
    df.isna().sum()
      .rename("n_manquants")
      .to_frame()
      .assign(pourcentage=lambda x: (100 * x["n_manquants"] / len(df)).round(2))
      .query("n_manquants > 0")
      .sort_values("n_manquants", ascending=False)
)
display(missing)

# Genre unknown documenté comme manquant (retiré avant analyse)
display(pd.DataFrame([UNKNOWN_MISSING]))


### 3. Doublons

On recherche les doublons sur toute la ligne et sur le couple **utilisateur-film**. Dans ce jeu de données, un même utilisateur ne doit noter un film qu'une seule fois.


In [ ]:
# Doublons : y a-t-il des lignes répétées ?
# (on exclut `genres` des doublons complets : une liste n'est pas hashable)
cols_hors_genres = [c for c in df.columns if c != "genres"]
duplicates = pd.DataFrame({
    "doublons_complets": [int(df.duplicated(subset=cols_hors_genres).sum())],
    "doublons_user_film": [int(df.duplicated(["user_id", "item_id"]).sum())],
})
display(duplicates)


### 4. Catégories et modalités

`nunique()` indique le nombre de modalités différentes et `value_counts()` leur répartition.
On vérifie ainsi les valeurs étranges dans le sexe et les notes.


In [ ]:
# Catégories : les valeurs sont-elles cohérentes ?
category_summary = pd.DataFrame({
    "colonne": ["gender", "rating"],
    "nunique": [
        int(df["gender"].nunique(dropna=True)),
        int(df["rating"].nunique(dropna=True)),
    ],
    "valeurs": [
        ", ".join(sorted(df["gender"].dropna().astype(str).unique())),
        ", ".join(str(value) for value in sorted(df["rating"].dropna().unique())),
    ],
})
display(category_summary)
display(df["gender"].value_counts(dropna=False).rename_axis("gender").to_frame("effectif"))
display(df["rating"].value_counts(dropna=False).sort_index().rename_axis("rating").to_frame("effectif"))


On constates que 74 252 notes sont attribuées par des hommes et 25 738 par des femmes. Cela représente respectivement 74,3 % et 25,7 % du total. On note donc un déséquilibre important entre les sexes dans le jeu de données.

On constate également que les notes sont comprises entre 1 et 5, ce qui est cohérent avec l'échelle de notation attendue. Et qu'il y a un bias positif car plus de 55 % des notes sont supérieures ou égales à 4. Cela peut s'expliquer par le fait que les utilisateurs ont tendance à noter les films qu'ils ont appréciés, et non ceux qu'ils n'ont pas aimés.

### 5. Min, max, quantiles et incohérences

Les statistiques descriptives permettent de repérer les valeurs impossibles ou surprenantes. Les contrôles comparent ensuite les résultats à des bornes plausibles pour les notes, les âges, les années et les timestamps.


In [ ]:
# Min, max et quantiles : valeurs impossibles ou surprenantes ?
numeric_columns = ["rating", "age", "timestamp", "release_year"]
numeric_summary = df[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
numeric_summary = numeric_summary[["min", "1%", "25%", "50%", "75%", "99%", "max"]]
display(numeric_summary)

range_checks = pd.DataFrame({
    "controle": [
        "notes hors de l'intervalle [1, 5]",
        "âges hors de l'intervalle plausible [1, 120]",
        "années de sortie hors [1888, année actuelle]",
        "timestamps négatifs",
        "dates converties invalides",
    ],
    "effectif": [
        int((df["rating"].notna() & ~df["rating"].between(1, 5)).sum()),
        int((df["age"].notna() & ~df["age"].between(1, 120)).sum()),
        int((df["release_year"].notna() & ~df["release_year"].between(1888, pd.Timestamp.now().year)).sum()),
        int((df["timestamp"].notna() & (df["timestamp"] < 0)).sum()),
        int(df["datetime"].isna().sum()),
    ],
})
display(range_checks)


### 6. Dates, unités et types

On vérifie les types de colonnes avec `df.dtypes` et les bornes temporelles. Le timestamp est exprimé en secondes Unix, tandis que `datetime` doit être une date exploitable.


In [ ]:
# Dates, unités et types : sont-ils cohérents ?
type_summary = df.dtypes.astype(str).rename("type").to_frame()
display(type_summary)

date_summary = pd.DataFrame({
    "indicateur": [
        "timestamp min",
        "timestamp max",
        "datetime min",
        "datetime max",
        "années de sortie manquantes",
    ],
    "valeur": [
        df["timestamp"].min(),
        df["timestamp"].max(),
        df["datetime"].min(),
        df["datetime"].max(),
        int(df["release_year"].isna().sum()),
    ],
})
display(date_summary)


### 7. Conclusion

- Manquants `isna` : surtout `video_release_date` (vide) et quelques `release_date` / `release_year`.
- Genre **`unknown`** : information manquante → **exclu** (un film sans genre utile ne peut pas entrer dans une reco par genres).
- Doublons, catégories, min/max et dates contrôlés.

→ **Étape suivante :** comprendre genres et profils (âge/sexe) pour éclairer la reco « film aimé → autres films ».


## Étape 3 — Genres et profils : ce qui éclaire la recommandation

**Question produit rappelée :** si une personne a aimé un film en 1997, quels autres films aurait-elle pu aimer ?

Ici on ne construit pas encore le modèle. On prépare ses **ingrédients** :

- les **genres** (similarité entre films, candidats avec ≥ 1 genre en commun) ;
- l'**âge** et le **sexe** (pour **pondérer** le score selon le profil).

**Indicateur principal :** `part_genre = notes du genre / notes du groupe` (en %).  
Ce n'est **pas** la note moyenne : on mesure ce qui est le plus souvent noté, pour calibrer des poids — pas pour dire « mieux aimé ».

1. Contexte (qui note, catalogue)
2. Corrélations entre genres (bundles / rayon des candidats)
3. Parts et notes selon le **sexe** (pondération)
4. Parts selon l'**âge** (pondération)
5. Synthèse — lien avec la reco


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

users = df.drop_duplicates("user_id")
films = df.drop_duplicates("movie_id")

df["age_bin"] = pd.cut(
    df["age"],
    bins=[0, 17, 24, 34, 44, 54, 100],
    labels=["≤17", "18-24", "25-34", "35-44", "45-54", "55+"],
)

print(f"Notes : {len(df):,}  |  Utilisateurs : {len(users):,}  |  Films : {len(films):,}")


### 1. Contexte — qui note, et quels genres existent ?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gender_counts = (
    users["gender"].map({"M": "Hommes", "F": "Femmes"})
    .value_counts()
    .reindex(["Hommes", "Femmes"])
)
gender_counts.plot(
    kind="bar",
    ax=axes[0],
    color=["#4C72B0", "#DD8452"],
)
axes[0].set_title("Nombre d'utilisateurs par sexe")
axes[0].set_xlabel("")
axes[0].set_ylabel("Effectif")
axes[0].tick_params(axis="x", rotation=0)
sns.histplot(users["age"], bins=20, ax=axes[1], color="darkorange")
axes[1].set_title("Répartition des âges")
plt.tight_layout()
plt.show()

catalogue_genres = (
    films["genres"].explode().value_counts().reindex(GENRES).fillna(0).astype(int)
)
catalogue_genres.sort_values().plot(kind="barh", figsize=(9, 5), color="slateblue")
plt.title("Catalogue — nombre de films par genre")
plt.xlabel("Nombre de films")
plt.tight_layout()
plt.show()


**Lecture**

- Panel déséquilibré : plus d'**hommes** que de **femmes** ; public surtout jeune adulte → une reco « moyenne » sera tirée vers les goûts masculins / jeunes si on ne pondère pas.
- Catalogue dominé par Drama / Comedy : beaucoup de films auront ces genres en commun → le filtre « ≥ 1 genre en commun » reste large ; la pondération et la similarité de genres devront affiner.


### 2. Corrélations entre genres

Les films ont souvent plusieurs genres. On regarde quels genres **apparaissent ensemble**
sur les mêmes films (corrélation de Pearson entre indicateurs 0/1 au niveau film).
Une forte corrélation positive = genres souvent associés dans le catalogue.


In [ ]:
# Indicateurs 0/1 par film, puis corrélations entre genres
genre_flags = pd.DataFrame({
    g: films["genres"].map(lambda gs, genre=g: int(genre in gs)) for g in GENRES
})
corr_genres = genre_flags.corr(method="pearson")

plt.figure(figsize=(11, 9))
sns.heatmap(
    corr_genres,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    cmap="RdBu_r",
    center=0,
    vmin=-0.5,
    vmax=0.5,
    square=True,
    linewidths=0.3,
)
plt.title("Corrélation entre genres (co-occurrence sur les films)")
plt.tight_layout()
plt.show()

# Paires les plus liées (hors diagonale)
pairs = (
    corr_genres.where(np.triu(np.ones(corr_genres.shape), k=1).astype(bool))
    .stack()
    .rename("corr")
    .reset_index()
    .rename(columns={"level_0": "genre_1", "level_1": "genre_2"})
    .sort_values("corr", ascending=False)
)
display(pairs.head(10).round(3))
display(pairs.tail(5).round(3))


**Lecture — genres les plus corrélés**

Les corrélations les plus fortes (positives) regroupent des **familles de films** cohérentes :

- **Animation ↔ Children's (~0.44)** : la paire la plus liée. Beaucoup de films d'animation sont aussi tagués pour enfants.
- **Action ↔ Adventure (~0.34)** et **Action ↔ Sci-Fi (~0.27)** : bloc « spectacle / blockbuster ».
- **Animation ↔ Musical (~0.33)** et **Children's ↔ Fantasy (~0.29)** : prolongent la famille jeunesse / conte.

À l'opposé, de fortes corrélations **négatives** montrent des genres qui cohabitent peu :

- **Comedy ↔ Drama (~−0.34)** et **Comedy ↔ Thriller (~−0.22)** : comédie peu associée au drame / thriller.
- **Action ↔ Drama (~−0.21)** : action et drame se croisent moins souvent dans le catalogue.

**Conséquence pour la suite :** quand on dit qu'un groupe note plus d'Action, on traîne souvent aussi Adventure / Sci-Fi ; idem Animation avec Children's. Les profils âge/sexe doivent se lire avec ces bundles en tête, pas genre par genre comme s'ils étaient indépendants.

**Pour la reco :** ces bundles justifient de chercher d'abord des films partageant au moins un genre avec le film aimé (et souvent tout un paquet lié : Action+Adventure, Animation+Children's…).


### 3. Parts de notes et notes moyennes selon le sexe

Deux lectures complémentaires :
1. **Parts** — quels genres sont le plus souvent notés (quantité) ;
2. **Notes moyennes** — comment ces genres sont notés (valeur 1–5).


In [ ]:
from matplotlib.patches import Patch

def preference_share(frame, group_col, genres=GENRES):
    """Part des notes d'un groupe qui touchent chaque genre (0–1), via la liste genres."""
    out = {}
    for key, sub in frame.groupby(group_col, observed=True):
        lists = sub["genres"]
        out[key] = {g: lists.map(lambda gs, genre=g: genre in gs).mean() for g in genres}
    return pd.DataFrame(out)


def mean_rating_by_genre(frame, group_col, genres=GENRES):
    """Note moyenne (1–5) des évaluations qui touchent chaque genre."""
    out = {}
    for key, sub in frame.groupby(group_col, observed=True):
        out[key] = {}
        for g in genres:
            mask = sub["genres"].map(lambda gs, genre=g: genre in gs)
            out[key][g] = sub.loc[mask, "rating"].mean()
    return pd.DataFrame(out)


color_femmes, color_hommes = "#DD8452", "#4C72B0"

# --- 1) Parts de notes (quantité) ---
pref_gender = preference_share(df, "gender")
tbl = pd.DataFrame({
    "Femmes_%": (100 * pref_gender["F"]).round(2),
    "Hommes_%": (100 * pref_gender["M"]).round(2),
})
tbl["ecart_pts"] = (tbl["Femmes_%"] - tbl["Hommes_%"]).round(2)
tbl = tbl.sort_values("ecart_pts", ascending=False)
display(tbl)

ecarts = tbl["ecart_pts"].sort_values()
lim = max(6.0, float(ecarts.abs().max()) + 1.5)
colors = [color_femmes if v > 0 else color_hommes for v in ecarts]

fig, ax = plt.subplots(figsize=(10, 6))
ecarts.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-lim, lim)
ax.set_xlabel("Écart de part de notes (points de %)")
ax.set_title("Parts de notes par genre : écart Femmes − Hommes")
ax.legend(
    handles=[
        Patch(facecolor=color_femmes, label="Femmes"),
        Patch(facecolor=color_hommes, label="Hommes"),
    ],
    loc="lower right",
    frameon=True,
)
for y, (genre, val) in enumerate(ecarts.items()):
    ax.text(
        val + (0.2 if val >= 0 else -0.2),
        y,
        f"{val:+.1f}",
        va="center",
        ha="left" if val >= 0 else "right",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

# --- 2) Notes moyennes (valeur 1–5) ---
# SYNC_OK 2026-10-19 15:13:31
mean_gender = mean_rating_by_genre(df, "gender")
tbl_notes = pd.DataFrame({
    "Hommes": mean_gender["M"].round(2),
    "Femmes": mean_gender["F"].round(2),
})
tbl_notes["ecart"] = (tbl_notes["Femmes"] - tbl_notes["Hommes"]).round(2)

# Tri unique : note des hommes (croissant pour barh : bas = moins aimé, haut = plus aimé)
ordre = tbl_notes["Hommes"].sort_values(ascending=True).index.tolist()
tbl_notes = tbl_notes.loc[reversed(ordre)]
display(tbl_notes)

hommes_vals = tbl_notes.loc[ordre, "Hommes"].tolist()
femmes_vals = tbl_notes.loc[ordre, "Femmes"].tolist()
y_pos = list(range(len(ordre)))

# Dumbbell : une ligne par genre relie la note Hommes et la note Femmes
fig, ax = plt.subplots(figsize=(10, 7))
for y, h, f in zip(y_pos, hommes_vals, femmes_vals):
    ax.plot([h, f], [y, y], color="0.75", linewidth=2, zorder=1)
ax.scatter(hommes_vals, y_pos, s=90, color=color_hommes, zorder=3, label="Hommes")
ax.scatter(femmes_vals, y_pos, s=90, color=color_femmes, zorder=3, label="Femmes")
ax.set_yticks(y_pos)
ax.set_yticklabels(ordre)
ax.set_xlabel("Note moyenne (1–5)")
ax.set_xlim(3.0, 4.2)
ax.set_title("Note moyenne par genre — Hommes vs Femmes (tri : note des hommes ↑)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Lecture**

- Échelle en **points de %** : un écart de +4 signifie que ce genre occupe 4 points de plus
  dans les notes des femmes que dans celles des hommes.
- À droite (positif) : genres relativement plus notés par les **femmes** sont souvent: la Romance (avec 22% et un ecart de 4.44) et le Drama avec (42% et un ecart de 3.87).
- À gauche (négatif) : genres relativement plus notés par les **hommes** sont souvent: l'Action (avec 27% et un ecart de -5.99),le Sci-Fi (avec 13% et un ecart de -3.39).
- Drama et Comedy restent élevés **pour les deux** (offre dominante) : l'écart relativise mieux que la part brute.

On peut en conclure que les femmes ont tendance à noter proportionnellement plus de films de Romance et de Drama, tandis que les hommes notent proportionnellement plus de films d'Action et de Sci-Fi.

**Pour la reco :** ces écarts peuvent servir de **poids** selon le sexe (ex. légèrement favoriser Romance/Drama pour un profil femme, Action/Sci-Fi pour un profil homme), sans remplacer la similarité de genres avec le film aimé.


### 4. Parts de notes par genre selon l'âge


In [ ]:
pref_age = preference_share(df, "age_bin")
# Parts en % pour une échelle lisible
pref_age_pct = (100 * pref_age).round(1)

top_genres = (
    df["genres"].explode().value_counts()
    .reindex(GENRES).fillna(0).astype(int)
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)
pref_age_top = pref_age_pct.loc[top_genres]
display(pref_age_top)

plt.figure(figsize=(10, 5))
sns.heatmap(pref_age_top, annot=True, fmt=".1f", cmap="YlOrRd", vmin=0, vmax=50)
plt.title("Part des notes par genre selon la tranche d'âge (%)")
plt.xlabel("Tranche d'âge")
plt.ylabel("Genre de film")
plt.tight_layout()
plt.show()

focus = ["Drama", "Action", "Comedy", "Romance", "Children's", "War"]
pref_age_pct.loc[focus].T.plot(marker="o", figsize=(10, 5))
plt.title("Évolution des parts de notes (%) selon l'âge")
plt.ylabel("Part des notes du groupe (%)")
plt.xlabel("Tranche d'âge")
plt.ylim(0, None)
plt.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


**Lecture**

- Heatmap (%): plus la case est chaude, plus le genre occupe une grande part des notes de la tranche.
- Avec l'âge, la part de **Drama** (et souvent War) augmente ; **Action** / **Comedy** / **Children's**
  baissent plutôt chez les plus âgés.
- Même indicateur que pour le sexe : structure des notes, pas satisfaction (note moyenne).

**Pour la reco :** l'âge peut moduler les mêmes poids de genres (ex. plus de Drama pour les profils plus âgés, moins d'Action / Children's).


### 5. Synthèse — vers la recommandation

| Ingrédient reco | Ce que l'EDA apporte |
| --- | --- |
| Film aimé → candidats | Les genres (et leurs corrélations) définissent un rayon naturel : films avec **≥ 1 genre en commun**, souvent dans le même bundle. |
| Pondération par le **sexe** | Romance et Drama : parts plus élevées chez les femmes ; Action, Sci-Fi et Thriller : plus élevées chez les hommes. |
| Pondération par l'**âge** | Drama ↑ avec l'âge ; Action (et Children's chez les plus jeunes) plus présentes chez les plus jeunes. |
| Ce qu'on ne conclut pas | Pas de jugement sur la note moyenne ni de causalité âge/sexe → goût. Les parts servent à **pondérer**, pas à prouver qu'un film sera aimé. |

→ **Étape suivante :** limites de ces constats avant de les utiliser dans un modèle.


## Étape 4 — Qu'est-ce qui manque ?

L'étape 3 a préparé une reco du type : *film aimé + genres (+ âge/sexe en pondération)*.

Ici : ce que l'EDA **ne permet pas** encore de garantir pour cette question.

1. Biais
2. Limites des données
3. Représentativité
4. Causalité
5. Contrôles d'interprétation
6. Conclusion


### 1. Biais

- **Biais de sélection** : les utilisateurs et les films de MovieLens ne représentent ni toute la population ni l'ensemble du catalogue mondial.
- **Biais de participation** : on voit seulement ce qui est noté, pas ce qui est regardé sans note.
- **Biais de popularité / catalogue** : Drama et Comedy sont partout ; une part élevée de notes
  pour Drama reflète aussi l'offre, pas seulement un goût intrinsèque.
- **Pour la reco :** un système calé sur ce panel peut reproduire ces biais (hommes / Drama-Comedy) si on ne les contrôle pas.


### 2. Limites des données

- Le jeu contient des notes, un profil utilisateur déclaré (âge, sexe) et des genres, mais **pas** :
  le contexte de visionnage, les recommandations reçues, les goûts non exprimés, ni la raison d'une note.
- Les données sont **historiques** (1997–1998) : les comportements et l'offre de films ont évolué depuis.
- Certaines modalités sont **peu représentées** (tranches d'âge rares, genres peu notés) : une moyenne
  calculée sur peu d'observations doit être lue avec prudence.
- `video_release_date` est vide ; quelques `release_date` / `release_year` manquent.


### 3. Représentativité (lien avec la reco)

Les poids âge/sexe dérivés ici décrivent **MovieLens 100k**, pas tous les spectateurs de 1997 ni d'aujourd'hui.

- Hommes sur-représentés → sans pondération explicite, la similarité « moyenne » colle à leurs genres.
- Petits effectifs (âges élevés, genres rares) → poids instables.
- « Plus noté » ≠ « plus aimé » : pondérer sur les parts aide le **routing** par genres, pas à prédire la note 1–5.


### 4. Causalité

Observer qu'un groupe note plus souvent un genre **ne prouve pas** qu'il faut lui recommander ce genre.  
L'offre, la période 1997–1998 et d'autres facteurs non observés interviennent.  
Les poids âge/sexe restent des **heuristiques** descriptives pour la reco, pas des lois causales.


### 5. Contrôles d'interprétation

Quelques indicateurs pour rappeler l'échelle du jeu et la fragilité des petits effectifs.


Effectifs et volumes (échelle du jeu) :


In [ ]:
print(f"Utilisateurs observés : {df['user_id'].nunique():,}")
print(f"Films observés        : {df['item_id'].nunique():,}")
print(f"Notes observées       : {len(df):,}")

notes_par_film = df.groupby("item_id").size()
notes_par_user = df.groupby("user_id").size()
print()
print(f"Films avec < 10 notes     : {(notes_par_film < 10).sum():,}")
print(f"Film le plus noté         : {notes_par_film.max():,} notes")
print(f"Utilisateur le plus actif : {notes_par_user.max():,} notes")

# Répartition hommes / femmes (volume de notes)
by_gender = (
    df.groupby("gender")
      .agg(n_notes=("rating", "size"), n_users=("user_id", "nunique"))
)
by_gender.index = by_gender.index.map({"M": "Hommes", "F": "Femmes"})
display(by_gender)


**Lecture**

- Le panel est déséquilibré : les **hommes** produisent nettement plus de notes que les femmes.
- Beaucoup de films ont peu de notes (< 10) : toute moyenne ou part sur ces titres est fragile.
- Ces volumes rappellent que les profils âge/sexe se lisent à l'échelle de **ce** jeu MovieLens, pas de toute la population.


### 6. Conclusion

- **Question produit :** si une personne a aimé un film en 1997, quels autres films aurait-elle pu aimer ?

- **Ce que l'EDA montre (utile à la reco) :**
  - Les **genres** structurent le catalogue (bundles corrélés : Animation–Children's, Action–Adventure/Sci-Fi…) → base naturelle pour filtrer les candidats avec **≥ 1 genre en commun**.
  - L'activité de notation varie selon le **sexe** et l'**âge** (parts de notes) : femmes plutôt Romance / Drama (écarts vers +4 pts) ; hommes plutôt Action / Sci-Fi / Thriller (Action vers −6 pts). Avec l'âge, la part de Drama augmente, celle d'Action diminue.
  - Ces écarts peuvent **pondérer** un score de reco basé sur les genres du film aimé.
  - L'exclusion du genre `unknown` et le travail en parts (%) évitent de traiter un manque comme un genre et limitent l'effet des volumes bruts.

- **Ce qu'elle ne montre pas :**
  - On mesure surtout le **volume** d'activité par genre, pas une satisfaction : la note moyenne reste proche (~3,5/5) entre groupes.
  - Pas de causalité âge/sexe → goût, ni un modèle de reco déjà entraîné, ni une généralisation hors MovieLens 1997–98.

- **Biais à garder :** panel hommes sur-représentés ; offre Drama / Comedy dominante ; données historiques.

- **Suite :** construire la reco (candidats à genres partagés + score pondéré âge/sexe) et l'évaluer.